In [1]:
"""
RAG 시스템에서 PDF나 긴 문서를 문단 단위로 쪼갤 때, 페이지의 끝과 다음 페이지의 시작이 겹치도록(=overlap) 하는 것은 문맥 보존을 위한 핵심 전략이다.
왜 overlap이 필요할까? 예를 들어보면
  Page 1 끝: 그는 아내를 찾아…
  Page 2 시작: 간신히 병원에 도착했다.
이걸 페이지 단위로 나눠서 LLM에 전달하면 다음과 같은 문제가 생긴다.
  Page 1만 본다면 “어디로 간 거지?”
  Page 2만 본다면 “누가 간 거야?”
즉, 문맥 단절 → 정확한 질의 응답 불가
이를 막기 위해 문단 또는 문장 단위의 겹침(=sliding window, overlap) 이 필요하다.
"""

!pip install langchain langchain-community pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
from langchain_community.document_loaders import PyMuPDFLoader

# 1. PyMuPDFLoader로 PDF 문서 로드
loader = PyMuPDFLoader("foods_kor.pdf")
documents = loader.load()
print(len(documents))   # 10
print(documents)
print(documents[0].page_content)
print(documents[0].metadata)

8
[Document(metadata={'producer': 'Microsoft® Office Word 2007', 'creator': 'Microsoft® Office Word 2007', 'creationdate': 'D:20250417045540', 'source': 'foods_kor.pdf', 'file_path': 'foods_kor.pdf', 'total_pages': 8, 'format': 'PDF 1.5', 'title': '', 'author': 'acorn', 'subject': '', 'keywords': '', 'moddate': 'D:20250417045540', 'trapped': '', 'modDate': 'D:20250417045540', 'creationDate': 'D:20250417045540', 'page': 0}, page_content='1 페이지 \n \n한국 요리에 대한 고찰 \n한국식 요리의 총칭. 한식(韓食)이라고 부르기도 한다. 주로 한국의 전통식 요리를 \n뜻하며, 현대적으로 재창조된 한식은 \'퓨전 한식\' 등으로도 불린다. \n한식진흥법에서는 한국에서 사용되어 온 식재료 또는 그와 유사한 식재료를 \n사용하여 한국 고유의 조리방법 또는 그와 유사한 조리방법을 이용하여 만들어진 \n음식과 그 음식과 관련된 유형·무형의 자원·활동 및 음식문화를 "한식"으로 \n정의하고 있다. (한식진흥법 제2 조 제1 호) \n \n1. 역사와 분포 \n다른 여느 나라가 그렇듯이 한식은 여러 가지 경로와 계층을 통해 발전했다. 가장 \n큰 갈래 위주로 분류하자면 궁중 음식 - 삼국시대 이후 중앙 집권이 굳혀지며 이어져 \n내려온 화려한 상차림. 특별히 치우쳐진 바 없이 여러 지역의 식재를 골라 다양하게 \n섞어 쓴다는 특징이 있다. 개성 한정식은 반가 음식이긴 하나 옛 수도였기에 궁중 \n음식의 화려한 특징을 가졌으며 수도권 위주로 왕실의 영향을 받아 실질적으로는 \n궁중 음식에 가깝다. \n\uf0b7 \n반가 음식 - 삼

In [3]:
# 2. 문단 단위 분리 (한 페이지 내 문장들이 하나의 document.page_content에 있음)
paras = []

for i, doc in enumerate(documents, start=1):
    page_num = doc.metadata.get("page", i)   # 1) 페이지 번호 확인
    print(f"페이지 {page_num} ===")
    print(doc.page_content[:100], "\n")  # 2) 페이지 제목/첫 부분 출력

    # 3) 문단 분리해서 누적
    paragraphs = [ p.strip() for p in doc.page_content.split('\n') if p.strip() ]
    paras.extend(paragraphs)

print("\n총 문단 수:", len(paras))
print(paras[:5])   # 처음 5개만 확인


# 기존 코드는 각 페이지의 문단을 \n 기준으로 쪼갠 뒤 독립적으로 처리하였다.
# 이제 페이지 간 문맥이 끊기지 않도록 이전 페이지의 마지막 문단 몇 개를 현재 페이지 앞에 겹쳐 붙이는(overlap) 방식으로 개선
# 문단 단위 분리 + 페이지 간 overlap 처리  -----------------------
texts = []
overlapped_texts = []    # overlap된 문단 저장용

overlap_count = 2
prev_paragraphs = []

for doc_index, doc in enumerate(documents):
    page_num = doc.metadata.get("page", -1)

    paragraphs = [p.strip() for p in doc.page_content.split('\n') if p.strip()]

    # 이전 페이지의 마지막 문단 일부를 앞에 붙임
    if prev_paragraphs:
        # 기록: 어떤 페이지에서 어떤 문단이 겹쳐졌는지 추적
        for p in prev_paragraphs:
            overlapped_texts.append({
                "from_page": page_num - 1,
                "to_page": page_num,
                "text": p
            })
        paragraphs = prev_paragraphs + paragraphs

    texts.extend(paragraphs)
    prev_paragraphs = paragraphs[-overlap_count:]

print(f"총 문단 수: {len(texts)}")
print("문단 예시(일부 문단 출력):", texts[:5])

# 겹쳐진(overlap) 문단만 출력
print("\n Overlap된 문단 목록:")
for i, item in enumerate(overlapped_texts):
    print(f"{i+1:02d}. [p.{item['from_page']} → p.{item['to_page']}] {item['text']}")

페이지 0 ===
1 페이지 
 
한국 요리에 대한 고찰 
한국식 요리의 총칭. 한식(韓食)이라고 부르기도 한다. 주로 한국의 전통식 요리를 
뜻하며, 현대적으로 재창조된 한식은 '퓨전 한식' 등으 

페이지 1 ===
2 페이지 
 
먹었으며, 조선 후기에 이르러 주막 문화의 발달과 함께 발전했다. 현재 우리가 
즐기는 대부분의 음식이다. 
 
오늘날 우리가 먹는 한정식의 유래는 여러 가지로 설 

페이지 2 ===
3 페이지 
 
수산물은 
생선류를 
비롯하여 새우, 소라, 굴, 해삼, 전복 등 
매우 
다양하고 
해조류도 미역,  김,  파래,  다시마 등 그 종류가 많으며 높고 깊은 산맥 

페이지 3 ===
4 페이지 
 
3. 중세 
고려 시대에는 고려가 불교국가인 탓으로 육식 문화가 쇠퇴하였다. 송나라 사신이 
왔을 때 고기를 올려야 했던 때가 있었는데 도축하는 방법이 실전되어 불 

페이지 4 ===
5 페이지 
 
청나라로의 최대 수출품도 소가죽이었다. 박제가의 북학의에서 전국에서 하루에 소 
5 백필이 도축된다는 기록도 존재한다. 당시 조선은 소가죽은 수출하지만 쇠고기는 
 

페이지 5 ===
6 페이지 
 
고기를 먹지 못하고 잔칫날이나 제사처럼 특별한 때를 제외하면 고기 요리는 흔히 
접하지 못하였다. 당장 구글로 1970
년대 밥상이라고 쳐보면 당대에 어떻게 
먹었 

페이지 6 ===
7 페이지 
 
어렵다보니 외국인 채식주의자가 이 정보에까지 도달하는 데에는 아직 많은 
애로사항이 있다.  
조선의 밥상은 밥, 국, 김치, 장류를 기본으로 추가되는 찬 수에 따 

페이지 7 ===
8 페이지 
 
쌀을 통한 식사량이 다른 두 나라보다 많아 열량을 채우기 위해 기름에 튀길 
필요성을 느끼지 못했거나, 상업이 발달하지 않았기에 돈 대신으로도 쓸 수 있는 
쌀[1 


총 문단 수: 259
['1 페이지', '한국 요리에 대한 고찰', '한국식 요리의 총칭. 한식(韓食)이라고 부르기도 한다. 주로 한국의 전통식 요리를', "뜻하며, 현대